In [2]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_path = './deepfake_kaggle/Train'
train_dataset = ImageFolder(root=train_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [8]:
print(f'Number of training samples: {len(train_dataset)}')
print(f'Number of classes: {len(train_dataset.classes)}')
print(f'Class names: {train_dataset.classes}')
print(f"Mapping: {train_dataset.class_to_idx}")

Number of training samples: 140002
Number of classes: 2
Class names: ['Fake', 'Real']
Mapping: {'Fake': 0, 'Real': 1}


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0 
    total = 0

    print(f'Epoch {epoch+1}/{epochs}')

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Loss: {running_loss/len(train_loader):.4f}, Accuracy: {correct/total:.4f}')
    
torch.save(model.state_dict(), 'simple_cnn_kaggle_model.pth')

Epoch 1/5


KeyboardInterrupt: 

In [16]:
val_path = './deepfake_kaggle/Validation'
val_dataset = ImageFolder(root=val_path, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = SimpleCNN().to(device)
model.load_state_dict(torch.load('simple_cnn_kaggle_model.pth', map_location=device))

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        output = model(inputs)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accurarcy: {acc * 100:.2f}%\n")

print("Report by Class:")
print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))

Validation Accurarcy: 86.34%

Report by Class:
              precision    recall  f1-score   support

        Fake       0.89      0.82      0.86     19641
        Real       0.84      0.90      0.87     19787

    accuracy                           0.86     39428
   macro avg       0.87      0.86      0.86     39428
weighted avg       0.87      0.86      0.86     39428



In [17]:
test_path = './deepfake_kaggle/Test'
test_dataset = ImageFolder(root=test_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = SimpleCNN().to(device)
model.load_state_dict(torch.load('simple_cnn_kaggle_model.pth', map_location=device))

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        output = model(inputs)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Test Accurarcy: {acc * 100:.2f}%\n")

print("Report by Class:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

Test Accurarcy: 81.61%

Report by Class:
              precision    recall  f1-score   support

        Fake       0.81      0.83      0.82      5492
        Real       0.83      0.80      0.81      5413

    accuracy                           0.82     10905
   macro avg       0.82      0.82      0.82     10905
weighted avg       0.82      0.82      0.82     10905

